# experiment_gating_value

## Reproducibility bootstrap
Run first. Resolves paths for the authors' Drive, a fresh Colab (clones the anon repo), or a local clone. No edits needed.

In [ ]:
# === Reproducibility bootstrap (public bundle) ===
# Resolves all paths for: (a) authors' Google Drive, (b) fresh Colab (clones repo),
# (c) local clone. Sets CODE_DIR, DATA_DIR, RESULTS_DIR, DATA_PATH. Run first; no edits needed.
import os, sys
from pathlib import Path

RESULTS_SUBFOLDER = "experiment_gating_value"
DATA_FILENAME = None   # None for synthetic experiments

def _resolve():
    try:
        import google.colab  # noqa: F401
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        dr = Path('/content/drive/MyDrive')
        if (dr/'GNAVAR'/'code'/'gnavar_core.py').exists():
            b = dr/'GNAVAR'; return b/'code', b/'data', b/'results'
        # Fresh Colab without the authors' Drive: the repo files must be present in the
        # session. Anonymous-review repos cannot be git-cloned, so upload the bundle:
        #   1) Download the ZIP from the Anonymous GitHub page (Download / ZIP button).
        #   2) In Colab, upload the ZIP via the Files pane, then in a cell run:
        #        !unzip -o your_bundle.zip
        #   3) %cd into the unzipped repo folder, then run this notebook.
        for cand in [Path('/content')/'ICDM-GNAVAR-EDAE', Path.cwd()]:
            if (cand/'src'/'gnavar_core.py').exists():
                return cand/'src', cand/'data', cand/'results'
        raise FileNotFoundError(
            'Repo files not found in the Colab session. Download the ZIP from the '
            'Anonymous GitHub page, upload and unzip it here, then %cd into the folder '
            'and re-run. See the repository README, Path B, Option B1.')
    except ImportError:
        repo = Path.cwd()
        while repo != repo.parent and not (repo/'verify_paper_numbers.py').exists():
            repo = repo.parent
        return repo/'src', repo/'data', repo/'results'

CODE_DIR, DATA_DIR, RESULTS_ROOT = _resolve()
sys.path.insert(0, str(CODE_DIR))
RESULTS_DIR = RESULTS_ROOT / RESULTS_SUBFOLDER
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = (DATA_DIR / DATA_FILENAME) if DATA_FILENAME else None
DRIVE_ROOT = str(CODE_DIR.parent.parent)  # back-compat for any cell referencing DRIVE_ROOT
print('CODE_DIR    =', CODE_DIR)
print('RESULTS_DIR =', RESULTS_DIR)
if DATA_PATH: print('DATA_PATH   =', DATA_PATH)
import numpy as np
import pandas as pd
import json, hashlib, datetime, time, itertools, platform
try:
    import torch
    import torch.nn as nn
except Exception:
    pass
from gnavar_core import *  # model, generator, fit/eval utils
# === end bootstrap ===


# Baseline Comparison: Does Gating Earn Its Place?

**Reviewer concern (both reviewers).** The headline synthetic comparison is G-NAVAR (multiplicative gates) vs. additive NAVAR. On data with multiplicative structure the additive model *cannot* represent interactions, so it loses by construction (the 16.8x result). That shows you need *interactions*, not that you need *gates*. The fair test adds a third model that CAN represent two-way interactions but is NOT gated.

**Third model: additive + pairwise (GA2M / functional-ANOVA order 2):** $y=\text{bias}+\sum_j f_j(x_j)+\sum_{(j,k)} h_{j,k}(x_j,x_k)$, with $h_{j,k}$ a joint MLP on the concatenated lag-blocks. It can represent $f_j(x_j)\,g_k(x_k)$ exactly (a special case of a general 2-way function), so if G-NAVAR wins it is NOT because the competitor cannot express the structure.

**Pre-committed criterion.** We report whatever it shows. Two informative axes:
1. **Held-out MSE** (the forecasting comparison reviewers asked for).
2. **Recovery** (the identifiability-relevant axis): does each model concentrate interaction signal on the TRUE interacting pairs, or smear it across spurious pairs? On the synthetic DGP the truth is known: source x2 modulated by x3, and source x4 by x5 -> after target-drop the true pairs are (0,1) and (2,3).

**Capacity fairness.** We size the additive+pairwise model to MATCH OR EXCEED G-NAVAR's parameter count (reported per run), so G-NAVAR cannot win by the competitor being starved.

Outputs: `results.csv` (per size x seed x model), `recovery.csv` (per-pair interaction mass), `metadata.json`, `story.txt`.

## Cell 1: Drive + imports

## Cell 2: Additive+pairwise (GA2M) baseline

In [ ]:
class AdditivePairwiseNAVAR(nn.Module):
    """Interaction-aware, NON-gated: y = bias + sum_j f_j(x_j) + sum_(j,k) h_{j,k}(x_j,x_k)."""
    def __init__(self, n_sources, K, hidden_dim, pairs, joint_hidden=None):
        super().__init__()
        self.n_sources=n_sources; self.K=K; self.pairs=list(pairs); self.npairs=len(self.pairs)
        self.bias=nn.Parameter(torch.zeros(1))
        self.bases=BatchedMLP(n_groups=n_sources, K=K, hidden_dim=hidden_dim)
        jh=joint_hidden or hidden_dim
        if self.npairs>0: self.joint=BatchedMLP(n_groups=self.npairs, K=2*K, hidden_dim=jh)
    def forward(self, X_lag):
        out=self.bias+self.bases(X_lag).sum(1)
        if self.npairs>0:
            blocks=torch.stack([torch.cat([X_lag[:,j,:],X_lag[:,k,:]],1) for (j,k) in self.pairs],1)
            out=out+self.joint(blocks).sum(1)
        return out
    @torch.no_grad()
    def pair_interaction_mass(self, X_lag, bins=12):
        """PURE-interaction mass per pair: variance of the joint term h_{j,k} AFTER
        ANOVA-centering (removing absorbed main effects of x_j and x_k via binned
        conditional-mean subtraction). Raw var(h) overstates interaction because an
        unconstrained MLP absorbs main effects; centering isolates the genuine 2-way part."""
        if self.npairs==0: return {}
        blocks=torch.stack([torch.cat([X_lag[:,j,:],X_lag[:,k,:]],1) for (j,k) in self.pairs],1)
        H=self.joint(blocks).detach().cpu().numpy()  # (B, npairs)
        out={}
        for i,(j,k) in enumerate(self.pairs):
            h=H[:,i].astype(float).copy()
            xj=X_lag[:,j,:].mean(1).cpu().numpy(); xk=X_lag[:,k,:].mean(1).cpu().numpy()
            h=h-h.mean()
            for v in (xj,xk):  # subtract conditional mean over each variable (remove its main effect)
                edges=np.linspace(v.min(),v.max(),bins); idx=np.digitize(v,edges)
                for b in np.unique(idx):
                    m=idx==b
                    if m.sum()>0: h[m]=h[m]-h[m].mean()
            out[(j,k)]=float(np.var(h))  # residual variance = pure interaction mass
        return out

def fit_addpair(Xl,y,n_sources,K,hidden_dim,pairs,joint_hidden,seed,epochs,lr=1e-3,l1=0.0):
    torch.manual_seed(seed); np.random.seed(seed)
    m=AdditivePairwiseNAVAR(n_sources,K,hidden_dim,pairs,joint_hidden).to(DEVICE)
    Xt=torch.from_numpy(Xl).float().to(DEVICE); yt=torch.from_numpy(y).float().to(DEVICE)
    opt=torch.optim.Adam(m.parameters(),lr=lr,weight_decay=1e-5)
    bs=512; n=len(yt)
    for ep in range(epochs):
        perm=torch.randperm(n,device=DEVICE)
        for i in range(0,n,bs):
            idx=perm[i:i+bs]; opt.zero_grad(); pred=m(Xt[idx]); loss=((pred-yt[idx])**2).mean()
            if l1>0 and m.npairs>0:
                blocks=torch.stack([torch.cat([Xt[idx][:,j,:],Xt[idx][:,k,:]],1) for (j,k) in m.pairs],1)
                loss=loss+l1*m.joint(blocks).abs().mean()
            loss.backward(); opt.step()
    return m
@torch.no_grad()
def mse_addpair(m,Xl,y):
    Xt=torch.from_numpy(Xl).float().to(DEVICE); yt=torch.from_numpy(y).float().to(DEVICE)
    return float(((m(Xt)-yt)**2).mean().detach())
def nparams(m): return sum(p.numel() for p in m.parameters())

class BlackBoxMLP(nn.Module):
    """Capacity-matched black-box forecaster: flatten the (n_sources,K) lag window
    -> dense layers -> target. Forecast MSE only; NO interpretability/recovery claim.
    Tests whether G-NAVAR sacrifices predictive power for interpretability."""
    def __init__(self, n_sources, K, hidden_dim, n_layers=2):
        super().__init__()
        inp=n_sources*K; mods=[nn.Linear(inp,hidden_dim), nn.Tanh()]
        for _ in range(n_layers-1): mods+=[nn.Linear(hidden_dim,hidden_dim), nn.Tanh()]
        mods+=[nn.Linear(hidden_dim,1)]
        self.net=nn.Sequential(*mods)
    def forward(self, X_lag):  # (B,n_sources,K)
        return self.net(X_lag.reshape(X_lag.shape[0],-1)).squeeze(-1)

def fit_mlp(Xl,y,n_sources,K,hidden_dim,seed,epochs,lr=1e-3):
    torch.manual_seed(seed); np.random.seed(seed)
    m=BlackBoxMLP(n_sources,K,hidden_dim).to(DEVICE)
    Xt=torch.from_numpy(Xl).float().to(DEVICE); yt=torch.from_numpy(y).float().to(DEVICE)
    opt=torch.optim.Adam(m.parameters(),lr=lr,weight_decay=1e-5); bs=512; n=len(yt)
    for ep in range(epochs):
        perm=torch.randperm(n,device=DEVICE)
        for i in range(0,n,bs):
            idx=perm[i:i+bs]; opt.zero_grad(); loss=((m(Xt[idx])-yt[idx])**2).mean(); loss.backward(); opt.step()
    return m
@torch.no_grad()
def mse_mlp(m,Xl,y):
    Xt=torch.from_numpy(Xl).float().to(DEVICE); yt=torch.from_numpy(y).float().to(DEVICE)
    return float(((m(Xt)-yt)**2).mean().detach())


## Cell 3: Config; capacity-match the pairwise model to G-NAVAR

In [ ]:
SIZES=[5000,25000,100000]; SEEDS=[0,1,2,3,4]; K=2; HID=32; EPOCHS=300
# DGP (1-based): x1=target; sources x2,x3,x4,x5. True edges: x2 modulated by x3, x4 modulated by x5.
# make_lag_tensor_runs(target_col=0) DROPS the target, so the source axis is [x2,x3,x4,x5] -> 0-based [0,1,2,3].
# Hence true interacting (source,modulator) pairs are (x2,x3)=(0,1) and (x4,x5)=(2,3).
TRUE_PAIRS=[(0,1),(2,3)]
cfg=Config(n_vars=5,K=K,hidden_dim=HID,n_epochs=EPOCHS,l1_lambda=0.005,triviality_threshold=0.001)
# determine joint_hidden so AdditivePairwise params >= GNAVAR params (no starvation)
_g=GNAVAR(4,K,HID); g_params=sum(p.numel() for p in _g.parameters())
pairs=list(itertools.combinations(range(4),2))
jh=HID
while sum(p.numel() for p in AdditivePairwiseNAVAR(4,K,HID,pairs,jh).parameters())<g_params: jh+=8
print(f'GNAVAR params={g_params}; additive+pairwise joint_hidden={jh} -> params={sum(p.numel() for p in AdditivePairwiseNAVAR(4,K,HID,pairs,jh).parameters())} (>= GNAVAR, no starvation)')
mlp_h=HID
while sum(p.numel() for p in BlackBoxMLP(4,K,mlp_h).parameters())<g_params: mlp_h+=8
print(f'black-box MLP hidden={mlp_h} -> params={sum(p.numel() for p in BlackBoxMLP(4,K,mlp_h).parameters())} (>= GNAVAR)')
print('all candidate pairs:',pairs,'| true pairs:',TRUE_PAIRS)

## Cell 4: Run -- three models x sizes x seeds

In [ ]:
RES=RESULTS_DIR/'results.csv'; REC=RESULTS_DIR/'recovery.csv'
def gnavar_top_pair(model,Xt):
    # which (source,modulator) gate is strongest, as an unordered pair
    best=None;bs=-1
    for j in range(4):
        for k in range(4):
            if k==j: continue
            s=gate_triviality_score(model,Xt,j,k)
            if s>bs: bs=s; best=tuple(sorted((j,k)))
    return best,bs
rows=[]; recs=[]
for T in SIZES:
  for seed in SEEDS:
    X=simulate_dgp(T,cfg,seed=seed); Xl,y=make_lag_tensor_runs(X,np.array([[0,len(X)]]),K=K,target_col=0)
    cut=int(len(y)*0.8); Xtr,ytr,Xte,yte=Xl[:cut],y[:cut],Xl[cut:],y[cut:]
    Xtr_t=torch.from_numpy(Xtr).float().to(DEVICE)
    # G-NAVAR
    g=fit_gnavar_from_lag_with_restarts(Xtr,ytr,cfg,seed=seed,n_restarts=1,verbose=False)
    g_mse=held_out_mse_gnavar_from_lag(g,Xte,yte); g_pair,_=gnavar_top_pair(g,Xtr_t)
    # additive
    a=fit_pairwise_from_lag_with_restarts(Xtr,ytr,cfg,seed=seed,n_restarts=1,verbose=False)
    a_mse=held_out_mse_pairwise_from_lag(a,Xte,yte)
    # additive + pairwise (GA2M) -- interaction-capable, role 3
    ap=fit_addpair(Xtr,ytr,4,K,HID,pairs,jh,seed=seed,epochs=EPOCHS,l1=0.005)
    ap_mse=mse_addpair(ap,Xte,yte); mass=ap.pair_interaction_mass(Xtr_t)
    tot=sum(mass.values())+1e-12; true_frac=sum(mass[p] for p in TRUE_PAIRS)/tot
    ap_top=max(mass,key=mass.get)
    # black-box MLP -- capacity-matched forecaster, MSE only (interpretability-tax test)
    mlp=fit_mlp(Xtr,ytr,4,K,mlp_h,seed=seed,epochs=EPOCHS); mlp_mse=mse_mlp(mlp,Xte,yte)
    rows.append({'T':T,'seed':seed,'gnavar_mse':g_mse,'additive_mse':a_mse,'addpair_mse':ap_mse,'mlp_mse':mlp_mse,
      'gnavar_top_pair':str(g_pair),'gnavar_recovers':g_pair in TRUE_PAIRS,
      'addpair_top_pair':str(ap_top),'addpair_recovers':ap_top in TRUE_PAIRS,
      'addpair_true_mass_frac':true_frac,'gnavar_params':g_params,'addpair_params':nparams(ap),
      'mlp_params':sum(p.numel() for p in mlp.parameters())})
    for p,v in mass.items(): recs.append({'T':T,'seed':seed,'pair':str(p),'is_true':p in TRUE_PAIRS,'interaction_mass':v})
    print(f'T={T} seed={seed}: MSE g={g_mse:.4f} add={a_mse:.4f} addpair={ap_mse:.4f} mlp={mlp_mse:.4f} | g_top={g_pair} ap_top={ap_top} g_recov={g_pair in TRUE_PAIRS} ap_recov={ap_top in TRUE_PAIRS}',flush=True)
pd.DataFrame(rows).to_csv(RES,index=False); pd.DataFrame(recs).to_csv(REC,index=False)
print('\nsaved',RES)

## Cell 5: Summary + story

In [ ]:
df=pd.read_csv(RES)
from collections import Counter
import math
def norm_entropy(counts):
    n=sum(counts.values()); ps=[v/n for v in counts.values() if v>0]
    H=-sum(p*math.log(p) for p in ps); k=len([v for v in counts.values() if v>0])
    return abs(H/math.log(k)) if k>1 else 0.0
# per-T MSE summary (now incl. black-box MLP)
g=df.groupby('T').agg(gnavar_mse=('gnavar_mse','mean'),additive_mse=('additive_mse','mean'),
  addpair_mse=('addpair_mse','mean'),mlp_mse=('mlp_mse','mean'),
  gnavar_recov=('gnavar_recovers','mean'),addpair_recov=('addpair_recovers','mean'),
  addpair_true_mass=('addpair_true_mass_frac','mean')).reset_index()
pd.set_option('display.width',220)
print(g.to_string(index=False))
# across-seed STABILITY of recovered top pair, per T, for the two interaction-capable models
print('\nAcross-seed top-pair stability (lower entropy = more stable; modal = most-frequent pair):')
stab=[]
for T,grp in df.groupby('T'):
    gc=Counter(grp['gnavar_top_pair']); ac=Counter(grp['addpair_top_pair'])
    stab.append({'T':T,'gnavar_modal':gc.most_common(1)[0][0],'gnavar_modal_frac':round(gc.most_common(1)[0][1]/len(grp),2),
      'gnavar_entropy':round(norm_entropy(gc),2),'addpair_modal':ac.most_common(1)[0][0],
      'addpair_modal_frac':round(ac.most_common(1)[0][1]/len(grp),2),'addpair_entropy':round(norm_entropy(ac),2)})
stab=pd.DataFrame(stab); print(stab.to_string(index=False))

L=['GATING-VALUE COMPARISON (reviewer-facing, 4 roles)','='*54,
   f'Capacity (params): G-NAVAR={df.gnavar_params.iloc[0]}, additive+pairwise={df.addpair_params.iloc[0]}, MLP={df.mlp_params.iloc[0]} (both competitors >= G-NAVAR).','',
   'Held-out MSE (mean over seeds) -- additive=structural floor, MLP=interpretability-tax test:']
for r in g.itertuples():
    L.append(f'  T={int(r.T):>6}: G-NAVAR={r.gnavar_mse:.4f}  additive={r.additive_mse:.4f}  additive+pairwise={r.addpair_mse:.4f}  black-box-MLP={r.mlp_mse:.4f}')
L.append('')
L.append('Interaction recovery + across-seed STABILITY (capacity != identifiability test):')
for r in stab.itertuples():
    L.append(f'  T={int(r.T):>6}: G-NAVAR modal-pair={r.gnavar_modal} ({r.gnavar_modal_frac}, entropy {r.gnavar_entropy}) | additive+pairwise modal-pair={r.addpair_modal} ({r.addpair_modal_frac}, entropy {r.addpair_entropy})')
L.append('')
L.append('FRAMING: additive tests whether additive structure suffices (it does not).')
L.append('Black-box MLP tests whether G-NAVAR sacrifices predictive power (interpretability tax).')
L.append('additive+pairwise tests whether interaction CAPACITY alone yields STABLE recovery.')
L.append('G-NAVAR claim: stable, identifiable recovery under the support conditions -- NOT beating every forecaster.')
s='\n'.join(L); (RESULTS_DIR/'story.txt').write_text(s); print('\n'+s)
def _sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for ch in iter(lambda:f.read(8192),b''): h.update(ch)
    return h.hexdigest()
import gnavar_core as _gc
json.dump({'timestamp':datetime.datetime.now(datetime.timezone.utc).isoformat(),'experiment':'gating_value',
  'gnavar_core_sha256':_sha(_gc.__file__),'sizes':SIZES,'seeds':SEEDS,'true_pairs':[list(p) for p in TRUE_PAIRS],
  'models':['additive(PairwiseNAVAR)','gnavar','additive+pairwise(GA2M)','blackbox_mlp'],
  'note':'4 reviewer roles: additive floor; MLP interpretability-tax; GA2M capacity-vs-identifiability (seed stability); G-NAVAR MSE+stability. GA2M+MLP capacity-matched >= G-NAVAR.'},
  open(RESULTS_DIR/'metadata.json','w'),indent=2)
